# Data Analysis & Modeling: Classify a Content Submission into Three Categories

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
This follows the same format as [Will a Player Return?](codesignal_roblox_retention_practice.ipynb): **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic.

## The interviewer prompt

Before a content submission is reviewed, predict one coarse category: **clean**, **spam**, or **abuse**. Treat this as a mutually exclusive three-class teaching task, not Roblox's actual moderation policy.

**Problem 1 (14 minutes):** inspect and clean two supplied tables, join metadata, and summarize class balance by content type.
**Problem 2 (21 minutes):** train multinomial logistic regression, inspect the three probabilities, evaluate per-class errors, and score a new submission.

This mock practices tabular ML. It does not ask you to train a text encoder. Provided numeric signals stand in for features computed at upload time.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the data |
| 9–14 min | Analyze and prepare features |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? What exactly is the target?

**Compared with Will a Player Return?** Keep the familiar cleaning, merge, stratified split and sklearn Pipeline. The main changes are **three labels instead of two**, **N-by-3 probabilities**, **argmax instead of a 0.5 threshold**, and **multiclass metrics**.

**Reference version.** Run cells top to bottom. Code is complete; each step includes checks and expandable follow-up answers. Try the matching practice notebook first.

## Data contract — supplied by the interviewer

The interviewer supplies **submissions.csv** (one row per submission, plus repeated export rows) and **creators.csv** (unique creator metadata). Data are synthetic. One unique creator per submission is deliberate, making a simple stratified split valid for this mock; real repeated creators may require a group split.

| Column | Meaning / policy |
| --- | --- |
| submission_id, creator_id | Join/identity columns, not numeric features |
| text_length | Number of characters at upload; negative sentinel means unknown |
| link_count | Number of links at upload |
| uppercase_ratio | Fraction of letters in uppercase, in [0,1] |
| repeated_token_ratio | Fraction of repeated tokens, in [0,1] |
| content_type | chat, description, or comment; may be missing |
| creators.account_age_days | Account age at upload; may be missing |
| label | Reviewer-provided class, available only after review |
| review_minutes | Time spent on the subsequent review; **leakage**, never a feature |

Every submission has a known label and matching creator. Drop exact duplicates; conflicting submission IDs should fail a check. Keep rows with incomplete predictors.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter and select **.venv-codesignal/bin/python**. No network, GPU or additional package installation is required.

The two input files are **data/codesignal_roblox_softmax/submissions.csv** and **creators.csv**. The provided cell imports libraries and sets DATA_DIR. Load both files with pd.read_csv in Step 1; do not modify the source CSVs.

Start the 35-minute coding timer after setup.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, f1_score, log_loss, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, precision_score, recall_score)
SEED = 42

DATA_DIR = Path("data/codesignal_roblox_softmax")
assert (DATA_DIR / "submissions.csv").exists(), "Run from the workspace root."
assert (DATA_DIR / "creators.csv").exists()


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Inspect submissions and creators. Count duplicate exports, invalid text lengths and missing fields. Inspect the label counts, but do not treat arbitrary category codes as ordered targets.

**Say before coding:** "This is single-label multiclass classification: each example belongs to exactly one of three classes."

**Possible interviewer questions**

1. Why not regression with clean=0, spam=1, abuse=2?
2. Could content actually be both spam and abuse?
3. What should you inspect before dropping missing rows?


In [2]:
submissions = pd.read_csv(DATA_DIR / "submissions.csv")
creators = pd.read_csv(DATA_DIR / "creators.csv")

print("submissions:", submissions.shape, "creators:", creators.shape)
submissions.info()
display(submissions.head(), creators.head(), submissions.isna().sum())
display(submissions["label"].value_counts())
print("exact duplicates:", submissions.duplicated().sum())
print("invalid length:", submissions["text_length"].lt(0).sum())


submissions: (1215, 9) creators: (1200, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   submission_id         1215 non-null   int64  
 1   creator_id            1215 non-null   int64  
 2   text_length           1215 non-null   float64
 3   link_count            1215 non-null   int64  
 4   uppercase_ratio       1215 non-null   float64
 5   repeated_token_ratio  1215 non-null   float64
 6   content_type          1175 non-null   object 
 7   label                 1215 non-null   object 
 8   review_minutes        1215 non-null   float64
dtypes: float64(4), int64(3), object(2)
memory usage: 85.6+ KB


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes
0,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918
1,1,1,505.0,2,0.223062,0.039823,comment,clean,1.170087
2,2,2,428.0,3,0.311449,0.349885,description,spam,5.320863
3,3,3,290.0,1,0.058440,0.084628,chat,spam,5.876839
4,4,4,287.0,1,0.479259,0.315910,description,clean,1.283389


,creator_id,account_age_days
0,0,1395.0
1,1,614.0
2,2,1387.0
3,3,124.0
4,4,480.0


submission_id            0
creator_id               0
text_length              0
link_count               0
uppercase_ratio          0
repeated_token_ratio     0
content_type            40
label                    0
review_minutes           0
dtype: int64

label
clean    713
abuse    286
spam     216
Name: count, dtype: int64

exact duplicates: 15
invalid length: 35


In [3]:
assert creators["creator_id"].is_unique
assert submissions["creator_id"].isin(creators["creator_id"]).all()
assert set(submissions["label"]) == {"clean", "spam", "abuse"}


<details>
<summary>Follow-up answers</summary>

**Why not regression with clean=0, spam=1, abuse=2?**

Those codes would invent distances and an ordering. Use a categorical target instead.

**Could content actually be both spam and abuse?**

Yes. That would require a multilabel contract, typically independent sigmoid outputs, not this mutually exclusive softmax exercise.

**What should you inspect before dropping missing rows?**

Check which columns are missing and what missingness means. Missing predictors do not automatically invalidate the label or the other fields.

</details>


## Problem 1 · Step 2 — Clean the data (4–9 min)

Create clean by dropping exact duplicates and replacing negative text_length with NaN. Left merge creators into frame with validate="many_to_one". Do not use review_minutes as a feature.

**Say before coding:** "The creator table has one row per key. Merge validation prevents accidental row multiplication."

**Possible interviewer questions**

1. Why many_to_one?
2. Why is review_minutes leakage?
3. Why postpone median imputation until after splitting?


In [4]:
clean = submissions.drop_duplicates().copy()
assert clean["submission_id"].is_unique
clean["text_length"] = clean["text_length"].mask(clean["text_length"].lt(0))
frame = clean.merge(creators, on="creator_id", how="left", validate="many_to_one")


In [5]:
assert len(frame) == submissions["submission_id"].nunique()
assert frame["text_length"].dropna().ge(0).all()
assert frame["submission_id"].is_unique


<details>
<summary>Follow-up answers</summary>

**Why many_to_one?**

The right-side creator key must be unique. The general submission schema allows many submissions per creator, although this fixture uses one.

**Why is review_minutes leakage?**

The review occurs after the moment we must produce the prediction.

**Why postpone median imputation until after splitting?**

A median is learned from data. Fit it on training rows only; the held-out rows must not influence preprocessing.

</details>


## Problem 1 · Step 3 — Analyze and prepare features (9–14 min)

Create class_fraction from the cleaned frame. Display counts by content_type and label, including missing content types. Identify the largest class and say why accuracy alone may hide poor minority recall.

**Say before coding:** "I will check each class instead of relying only on the overall score."

**Possible interviewer questions**

1. What does normalize=True change?
2. Should we immediately balance the dataset?
3. How do you check that a merge did not multiply rows?


In [6]:
class_fraction = frame["label"].value_counts(normalize=True)
display(class_fraction)
display(frame.groupby(["content_type", "label"], dropna=False).size().unstack(fill_value=0))


label
clean    0.584167
abuse    0.237500
spam     0.178333
Name: proportion, dtype: float64

label,abuse,clean,spam
content_type,,,
chat,94,204,69
comment,107,243,61
description,78,226,78
NaN,6,28,6


In [7]:
np.testing.assert_allclose(class_fraction.sum(), 1)
assert len(class_fraction) == 3


<details>
<summary>Follow-up answers</summary>

**What does normalize=True change?**

Counts become fractions of the nonmissing labels.

**Should we immediately balance the dataset?**

No. Start with a baseline and per-class metrics. Discuss business costs, class weights and probability calibration before changing the training distribution.

**How do you check that a merge did not multiply rows?**

Declare the expected key relationship with validate, then check row counts and the training-example identifier.

</details>


## Problem 2 · Step 4 — Select features and split (14–18 min)

Use text_length, link_count, uppercase_ratio, repeated_token_ratio, account_age_days as numeric_features and content_type as categorical_features. Create an 80/20 stratified split with SEED.

**Say before coding:** "Stratification approximately preserves the three-class proportions; preprocessing still fits only on train."

**Possible interviewer questions**

1. Does stratification make the classes equally common?
2. What if creators appear in both splits in production data?
3. How do you check that X and y still correspond?


In [8]:
numeric_features = ["text_length", "link_count", "uppercase_ratio",
                    "repeated_token_ratio", "account_age_days"]
categorical_features = ["content_type"]
X = frame[numeric_features + categorical_features]
y = frame["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)


In [9]:
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index)
assert not {"label", "review_minutes", "creator_id", "submission_id"} & set(X.columns)
assert y_train.nunique() == y_test.nunique() == 3


<details>
<summary>Follow-up answers</summary>

**Does stratification make the classes equally common?**

No. It preserves their original proportions approximately.

**What if creators appear in both splits in production data?**

For evaluating unseen creators, split by creator and verify disjoint groups. For future submissions, use a time-aware split matched to the intended deployment.

**How do you check that X and y still correspond?**

Select both from the same ordered table and apply the same split indices. Verify their indices match within each subset.

</details>


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Use the familiar numeric and categorical pipelines. Create model with LogisticRegression(solver="lbfgs", max_iter=500), and baseline with DummyClassifier(strategy="prior"). Fit both on train only.

**Say before coding:** "With three classes, this solver fits multinomial logistic regression: one logit per class, followed by softmax."

**Possible interviewer questions**

1. What is the model mathematically?
2. Does each class probability use an independent sigmoid?
3. What happens when you call model.fit on the Pipeline?


In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
model = Pipeline([("preprocess", preprocess),
                  ("classifier", LogisticRegression(solver="lbfgs", max_iter=500))])
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
print("class order:", model.classes_)


class order: ['abuse' 'clean' 'spam']


In [11]:
assert model.named_steps["classifier"].coef_.shape[0] == 3
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())


<details>
<summary>Follow-up answers</summary>

**What is the model mathematically?**

For transformed X of shape (N,D), logits Z = X @ W.T + b have shape (N,3). Softmax normalizes exp(Z) across each row. The objective is multiclass cross-entropy plus regularization.

**Does each class probability use an independent sigmoid?**

No. Softmax probabilities compete and sum to one. Independent sigmoids fit multilabel outcomes instead.

**What happens when you call model.fit on the Pipeline?**

The preprocessing steps fit and transform the training features. The final estimator then fits those transformed features against the training labels.

</details>


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create prob and pred. Display the first five probability rows with model.classes_ as column names. Evaluate model and baseline with accuracy, macro-F1 and log loss. Display classification_report and a labeled 3x3 confusion matrix.

**Say before coding:** "Argmax selects one class. Macro-F1 weights classes equally; log loss also evaluates confidence."

**Possible interviewer questions**

1. Why not threshold every probability at 0.5?
2. Why does log loss need probabilities?
3. Can you keep changing the model after looking at test metrics?


In [12]:
prob = model.predict_proba(X_test)
pred = model.predict(X_test)
display(pd.DataFrame(prob[:5], columns=model.classes_))
def evaluate_multiclass(estimator):
    p = estimator.predict_proba(X_test)
    labels = estimator.predict(X_test)
    return {"accuracy": accuracy_score(y_test, labels),
            "macro_f1": f1_score(y_test, labels, average="macro", zero_division=0),
            "log_loss": log_loss(y_test, p, labels=estimator.classes_)}
scores = pd.DataFrame({"prior": evaluate_multiclass(baseline),
                       "softmax": evaluate_multiclass(model)}).T
display(scores)
print(classification_report(y_test, pred, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=model.classes_),
                     index=["actual: " + c for c in model.classes_],
                     columns=["predicted: " + c for c in model.classes_]))


,abuse,clean,spam
0,0.326322,0.578746,0.094932
1,0.120457,0.755775,0.123768
2,0.084683,0.388527,0.526790
3,0.058309,0.525935,0.415756
4,0.138329,0.810129,0.051542


,accuracy,macro_f1,log_loss
prior,0.583333,0.245614,0.963912
softmax,0.620833,0.493873,0.882927


              precision    recall  f1-score   support

       abuse       0.39      0.23      0.29        57
       clean       0.66      0.86      0.75       140
        spam       0.60      0.35      0.44        43

    accuracy                           0.62       240
   macro avg       0.55      0.48      0.49       240
weighted avg       0.59      0.62      0.59       240



,predicted: abuse,predicted: clean,predicted: spam
actual: abuse,13,40,4
actual: clean,13,121,6
actual: spam,7,21,15


In [13]:
assert prob.shape == (len(y_test), 3)
np.testing.assert_allclose(prob.sum(axis=1), 1)
np.testing.assert_array_equal(pred, model.classes_[prob.argmax(axis=1)])
true_col = pd.Index(model.classes_).get_indexer(y_test)
np.testing.assert_allclose(scores.loc["softmax", "log_loss"],
    -np.log(prob[np.arange(len(y_test)), true_col]).mean())


<details>
<summary>Follow-up answers</summary>

**Why not threshold every probability at 0.5?**

With three classes, the highest probability may be below 0.5. Argmax still picks exactly one class under this contract.

**Why does log loss need probabilities?**

It penalizes assigning low probability to the true class, especially confident wrong predictions. Accuracy discards that confidence information.

**Can you keep changing the model after looking at test metrics?**

Use a validation split or cross-validation for those choices. Repeatedly selecting against the test set makes its reported performance optimistic.

</details>


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

Create new_submission with a missing account age and unseen content_type="profile_bio". Produce new_prob and new_label. Explain how you would handle uncertain predictions without treating this toy model as a production moderation system.

**Say before coding:** "The model returns a category and a confidence distribution; deployment needs a separate reviewed policy."

**Possible interviewer questions**

1. Does a probability of 0.8 guarantee 80% correctness?
2. How might we use a human review queue?
3. What must training and inference share?


In [14]:
new_submission = pd.DataFrame([{
    "text_length": 120, "link_count": 2, "uppercase_ratio": 0.1,
    "repeated_token_ratio": 0.2, "account_age_days": np.nan,
    "content_type": "profile_bio",
}])
new_prob = model.predict_proba(new_submission)
new_label = model.predict(new_submission)
display(pd.DataFrame(new_prob, columns=model.classes_))
print("Predicted category:", new_label[0])


,abuse,clean,spam
0,0.066891,0.511513,0.421596


Predicted category: clean


In [15]:
assert new_prob.shape == (1, 3)
np.testing.assert_allclose(new_prob.sum(axis=1), 1)


<details>
<summary>Follow-up answers</summary>

**Does a probability of 0.8 guarantee 80% correctness?**

No. Check calibration on held-out data. Synthetic performance is not evidence of real-world moderation reliability.

**How might we use a human review queue?**

Define an abstention or risk-based review policy, tune it on validation data for coverage and error costs, and audit performance across groups.

**What must training and inference share?**

The feature definitions, column meanings and fitted preprocessing. Reuse the saved pipeline rather than relearning imputation, scaling or categories.

</details>


## If the interviewer extends the problem

**Optional 5-minute mathematical follow-up:** reconstruct predict_proba from logits. In the reference, the next cell verifies it numerically. This is not required in the 35-minute exercise.

For one row: subtract its maximum logit, exponentiate, then divide by the row sum. Subtracting a constant leaves softmax unchanged and avoids overflow.


In [16]:
transformed = model.named_steps["preprocess"].transform(X_test)
classifier = model.named_steps["classifier"]
logits = transformed @ classifier.coef_.T + classifier.intercept_
shifted = logits - logits.max(axis=1, keepdims=True)
exp_logits = np.exp(shifted)
manual_prob = exp_logits / exp_logits.sum(axis=1, keepdims=True)
np.testing.assert_allclose(manual_prob, prob)
print("X:", transformed.shape, "W:", classifier.coef_.shape,
      "logits:", logits.shape, "probabilities:", manual_prob.shape)


X: (240, 8) W: (3, 8) logits: (240, 3) probabilities: (240, 3)


## Scope and sources

These are **original synthetic exercises**, not reported Roblox questions and not official CodeSignal assessments. The user-supplied **Data Analysis & Modeling Coding Interview** guide defines the format: 5 minutes intro, 35 minutes for two problems (analysis and modeling), 5 minutes Q&A; pandas, NumPy, sklearn and SciPy are available. The per-step times here are suggested practice checkpoints, not official grading criteria.

The guide does not identify a team or prescribe regression/softmax topics. We chose those to practice task-appropriate modeling and metrics. Business themes are informed by the [official PhD Early Career MLE posting](https://careers.roblox.com/jobs/8242623), which mentions engagement, safety and recommendation systems. Source checked 2026-10-22; exact interview questions are unknown.

Implementation references: [sklearn 1.6 LogisticRegression](https://scikit-learn.org/1.6/modules/generated/sklearn.linear_model.LogisticRegression.html), [model evaluation](https://scikit-learn.org/1.6/modules/model_evaluation.html), [data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage).
